# Data Architecture

We initialize a local SQLite database file and seed it with realistic telemetry data from 3 welding machines. It simulates voltage spikes, overheating, and normal operations. 

In [1]:
import sqlite3
import random
from datetime import datetime, timedelta

def setup_database():
    # Connect to local SQLite file
    conn = sqlite3.connect('factory.db')
    cursor = conn.cursor()
    
    # 1. Create Tables
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Machine_Logs (
            log_id INTEGER PRIMARY KEY AUTOINCREMENT,
            machine_id TEXT,
            voltage REAL,
            current REAL,
            wire_feed_speed REAL,
            temperature REAL,
            timestamp TEXT
        )
    ''')
    
    # 2. Generate and Insert Mock Industrial Data
    machines = ['Warrior-500i', 'Rogue-ET', 'Aristo-500ix']
    start_time = datetime.now() - timedelta(days=5)
    
    print("Generating simulated manufacturing logs...")
    
    # Generate records at 5-minute intervals
    for i in range(1440):  
        timestamp = (start_time + timedelta(minutes=5 * i)).strftime('%Y-%m-%d %H:%M:%S')
        for machine in machines:
            # Baseline parameters
            voltage = random.uniform(22.0, 38.0)
            current = random.uniform(150.0, 300.0)
            wire_feed_speed = random.uniform(10.0, 18.0)
            temperature = random.uniform(45.0, 75.0) # Normal operating temp
            
            # Inject a few intentional failures/anomalies for the AI to find
            if machine == 'Warrior-500i' and i in range(200, 215):
                voltage = random.uniform(45.0, 55.0)  # Critical Voltage Spike
                temperature = random.uniform(95.0, 110.0) # Dangerous overheating
                
            if machine == 'Rogue-ET' and i in range(800, 810):
                wire_feed_speed = random.uniform(1.0, 3.0) # Feeding motor jam
                temperature = random.uniform(85.0, 95.0)
                
            cursor.execute('''
                INSERT INTO Machine_Logs (machine_id, voltage, current, wire_feed_speed, temperature, timestamp)
                VALUES (?, ?, ?, ?, ?, ?)
            ''', (machine, voltage, current, wire_feed_speed, temperature, timestamp))
            
    conn.commit()
    conn.close()
    print("Database built successfully as 'factory.db'!")

if __name__ == "__main__":
    setup_database()


Generating simulated manufacturing logs...
Database built successfully as 'factory.db'!
